In [2]:
import joblib

X_train = joblib.load(
    "X_train_random.pkl"
)

X_test = joblib.load(
    "X_test_random.pkl"
)

y_train = joblib.load(
    "y_train_random.pkl"
)

y_test = joblib.load(
    "y_test_random.pkl"
)

In [3]:
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)    
print(y_test.shape)

(7984, 2453)
(1996, 2453)
(7984,)
(1996,)


In [14]:
from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer

from sklearn.preprocessing import RobustScaler

from sklearn.feature_selection import (
    VarianceThreshold,
    SelectKBest,
    mutual_info_regression
)

In [15]:
preprocessor_robust = Pipeline([

    (
        "imputer",
        SimpleImputer(strategy="mean")
    ),

    (
        "variance_filter",
        VarianceThreshold(threshold=0.01)
    ),

    (
        "scaler",
        RobustScaler()
    ),

    (
        "feature_selection",
        SelectKBest(
            score_func=mutual_info_regression,
            k=1000
        )
    )

])

In [16]:
from lightgbm import LGBMRegressor

from xgboost import XGBRegressor

from catboost import CatBoostRegressor

from sklearn.ensemble import RandomForestRegressor

from sklearn.svm import SVR

from sklearn.ensemble import ExtraTreesRegressor

from sklearn.ensemble import HistGradientBoostingRegressor

from sklearn.neural_network import MLPRegressor

In [18]:

# MODEL PIPELINES


lgbm_pipeline = Pipeline([

    ("preprocessor_robust", preprocessor_robust),

    (
        "model",
        LGBMRegressor(
            n_estimators=300,
            random_state=42
        )
    )

])


xgb_pipeline = Pipeline([

    ("preprocessor_robust", preprocessor_robust),

    (
        "model",
        XGBRegressor(
            n_estimators=300,
            random_state=42
        )
    )

])


cat_pipeline = Pipeline([

    ("preprocessor_robust", preprocessor_robust),

    (
        "model",
        CatBoostRegressor(
            iterations=300,
            verbose=0,
            random_state=42,
            allow_writing_files=False
        )
    )

])


rf_pipeline = Pipeline([

    ("preprocessor_robust", preprocessor_robust),

    (
        "model",
        RandomForestRegressor(
            n_estimators=300,
            random_state=42,
            n_jobs=-1, 
            max_depth=6
        )
    )

])

svr_pipeline = Pipeline([

    ("preprocessor_robust", preprocessor_robust),

    (
        "model",
        SVR(
            kernel="rbf",
            C=10,
            epsilon=0.1
        )
    )

])

mlp_pipeline = Pipeline([
    
    ('preprocessor_robust', preprocessor_robust),
    (
        'model',MLPRegressor (
    
    
                hidden_layer_sizes = (256,128),
                activation = 'relu',
                solver = 'adam',
                alpha = 0.0001,
                batch_size = 'auto',
                learning_rate = 'adaptive',
                learning_rate_init = 0.0001,
                max_iter = 500,
                early_stopping= True,
                random_state = 42
        )
        
    )
    
])

et_pipeline = Pipeline([

    ("preprocessor_robust", preprocessor_robust),

    (
        "model",
        ExtraTreesRegressor(

            n_estimators=300,

            random_state=42,

            n_jobs=-1,

            max_depth=10

        )
    )

])

hist_pipeline = Pipeline([

    ("preprocessor_robust", preprocessor_robust),

    (
        "model",
        HistGradientBoostingRegressor(

            max_iter=300,

            learning_rate=0.05,

            max_depth=6,

            random_state=42

        )
    )

])



# STORE ALL MODELS


models = {

    "LightGBM": lgbm_pipeline,

    "XGBoost": xgb_pipeline,

    "CatBoost": cat_pipeline,

    "RandomForest": rf_pipeline,
    
    "SVR": svr_pipeline,
    
    "MLPRegressor": mlp_pipeline,
    
    "ExtraTreesRegressor": et_pipeline, 
    
    "HistGradientBoostingRegressor": hist_pipeline
    

}


models = {

    "LightGBM": lgbm_pipeline,

    "XGBoost": xgb_pipeline,

    "CatBoost": cat_pipeline,

    "RandomForest": rf_pipeline,

    "ExtraTrees": et_pipeline,

    "HistGradient": hist_pipeline,

    "SVR": svr_pipeline,

    "MLPRegressor": mlp_pipeline

}

In [19]:
from sklearn.model_selection import cross_val_score

results = []

for model_name, pipeline in models.items():

    cv_scores = cross_val_score(

        pipeline,

        X_train,

        y_train,

        cv=5,

        scoring="r2",

        n_jobs=-1

    )

    results.append({

        "Model": model_name,

        "CV Mean R2": cv_scores.mean(),

        "CV Std": cv_scores.std()

    })

   
    print(model_name)
   

    print("CV Scores:")
    print(cv_scores)

    print("\nMean CV R2:")
    print(cv_scores.mean())

LightGBM
CV Scores:
[0.79354592 0.78451934 0.76326659 0.80074156 0.77015908]

Mean CV R2:
0.7824464982393186
XGBoost
CV Scores:
[0.76694746 0.76905977 0.72458948 0.76975398 0.74041164]

Mean CV R2:
0.75415246625722
CatBoost
CV Scores:
[0.78904135 0.78891635 0.75893395 0.80086688 0.76559594]

Mean CV R2:
0.7806708948739713
RandomForest
CV Scores:
[0.74846787 0.7307547  0.70871664 0.73701923 0.70712885]

Mean CV R2:
0.7264174582573146
ExtraTrees
CV Scores:
[0.78389142 0.76199517 0.74943512 0.77105919 0.74112029]

Mean CV R2:
0.7615002378487068
HistGradient
CV Scores:
[0.79689365 0.78791668 0.77097093 0.80582245 0.77256322]

Mean CV R2:
0.786833385724585
SVR
CV Scores:
[0.64035621 0.6490459  0.60760392 0.41142921 0.60537585]

Mean CV R2:
0.5827622164767431
MLPRegressor
CV Scores:
[ 3.72516800e-01  6.52461796e-01  6.79904675e-01 -1.05409881e+05
  4.20549880e-01]

Mean CV R2:
-21081.551167603313


In [20]:
import pandas as pd

In [21]:
results_df = pd.DataFrame(results)

results_df = results_df.sort_values(

    by="CV Mean R2",

    ascending=False

)

results_df

,Model,CV Mean R2,CV Std
5,HistGradient,0.786833,0.013552
0,LightGBM,0.782446,0.014008
2,CatBoost,0.780671,0.015784
4,ExtraTrees,0.761500,0.015191
1,XGBoost,0.754152,0.018396
3,RandomForest,0.726417,0.016142
6,SVR,0.582762,0.087400
7,MLPRegressor,-21081.551168,42164.165052


In [22]:
from sklearn.metrics import (

    r2_score,
    mean_absolute_error,
    mean_squared_error

)

In [23]:
import numpy as np

In [24]:
from sklearn.metrics import (

    r2_score,
    mean_absolute_error,
    mean_squared_error

)

test_results = []

for model_name, pipeline in models.items():

    # TRAIN MODEL
  

    pipeline.fit(
        X_train,
        y_train
    )



    
    # TRAIN PREDICTIONS
    

    y_train_pred = pipeline.predict(
        X_train
    )



    
    # TEST PREDICTIONS
    

    y_test_pred = pipeline.predict(
        X_test
    )



    
    # TRAIN METRICS
    

    train_r2 = r2_score(
        y_train,
        y_train_pred
    )

    train_rmse = np.sqrt(
        mean_squared_error(
            y_train,
            y_train_pred
        )
    )



    
    # TEST METRICS
    

    test_r2 = r2_score(
        y_test,
        y_test_pred
    )

    test_rmse = np.sqrt(
        mean_squared_error(
            y_test,
            y_test_pred
        )
    )

    test_mae = mean_absolute_error(
        y_test,
        y_test_pred
    )



    
    # STORE RESULTS
    

    test_results.append({

        "Model": model_name,

        "Train_R2": train_r2,

        "Test_R2": test_r2,

        "Train_RMSE": train_rmse,

        "Test_RMSE": test_rmse,

        "Test_MAE": test_mae

    })



    
    # PRINT RESULTS
    

    print("\n")
    print(model_name)
    

    print("Train R2 :", train_r2)
    print("Test R2  :", test_r2)

    print("Train RMSE :", train_rmse)
    print("Test RMSE  :", test_rmse)

    print("Test MAE :", test_mae)


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.072759 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 242289
[LightGBM] [Info] Number of data points in the train set: 7984, number of used features: 1000
[LightGBM] [Info] Start training from score -2.889159


c:\Users\jeshw\anaconda3\envs\solubility_env\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\jeshw\anaconda3\envs\solubility_env\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(




LightGBM
Train R2 : 0.9558973804517872
Test R2  : 0.8125184942009874
Train RMSE : 0.4993256120939943
Test RMSE  : 1.0081897475648998
Test MAE : 0.6557644708175397


XGBoost
Train R2 : 0.9713356994162254
Test R2  : 0.7946637693645581
Train RMSE : 0.40255231153397536
Test RMSE  : 1.0551054174228116
Test MAE : 0.683839552064867


CatBoost
Train R2 : 0.9138333540409049
Test R2  : 0.8078084893567535
Train RMSE : 0.6979450705901129
Test RMSE  : 1.0207753180069343
Test MAE : 0.6840296300505418


RandomForest
Train R2 : 0.7752294407000027
Test R2  : 0.7476722163779933
Train RMSE : 1.1272529051436766
Test RMSE  : 1.1696221835768978
Test MAE : 0.8344268847690505


ExtraTrees
Train R2 : 0.8870569274552569
Test R2  : 0.7906764789780721
Train RMSE : 0.7990637870997758
Test RMSE  : 1.0653003652152104
Test MAE : 0.7396066094818124


HistGradient
Train R2 : 0.9187386594313887
Test R2  : 0.8163418572388194
Train RMSE : 0.6777876346227298
Test RMSE  : 0.9978566463117836
Test MAE : 0.6562292742843147



In [25]:
results_df = pd.DataFrame(
    test_results
)

results_df = results_df.sort_values(

    by="Test_R2",

    ascending=False

)

results_df

,Model,Train_R2,Test_R2,Train_RMSE,Test_RMSE,Test_MAE
5,HistGradient,0.918739,0.816342,0.677788,0.997857,0.656229
0,LightGBM,0.955897,0.812518,0.499326,1.008190,0.655764
2,CatBoost,0.913833,0.807808,0.697945,1.020775,0.684030
1,XGBoost,0.971336,0.794664,0.402552,1.055105,0.683840
4,ExtraTrees,0.887057,0.790676,0.799064,1.065300,0.739607
3,RandomForest,0.775229,0.747672,1.127253,1.169622,0.834427
6,SVR,0.654765,0.663730,1.397041,1.350227,0.975150
7,MLPRegressor,0.824694,0.404318,0.995519,1.797093,0.870594
